In [1]:
# ============================================================
# CELL 1: Imports + Modell laden (identisch zu deinem Setup)
# ============================================================
import os
import torch
import openpyxl
import numpy as np
from PIL import Image, UnidentifiedImageError
from pathlib import Path
import mobileclip
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Gerät: {device}")

weights_path = r"C:\Users\Lukas\masterthesis\thesis_2026\data\weights\mobileclip_weights\mobileclip_s1.pt"

model, _, preprocess = mobileclip.create_model_and_transforms(
    'mobileclip_s1',
    pretrained=weights_path,
    device=device
)
model.eval()
print("Modell geladen: MobileCLIP-S1")


# ============================================================
# CELL 2: Gold Standard aus Excel laden (Spalte A + I)
# ============================================================
EXCEL_PATH = r"C:\Users\Lukas\Downloads\23062026_annotations_human_non_human.xlsx"
IMAGE_DIR = Path(r"C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Human_Non_Human\random_sample_images_human_non_human")

wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
ws = wb.active
rows = list(ws.iter_rows(min_row=2, values_only=True))

def norm(x):
    if x is None: return None
    return x.strip().lower().replace("-", "_")

filenames, labels = [], []
for r in rows:
    fname = r[0]        # Spalte A: filename
    gold = norm(r[8])   # Spalte I: gold_standard_final
    if fname is None or gold is None:
        continue
    filenames.append(fname)
    labels.append(gold)

print(f"Geladen: {len(filenames)} Bilder mit Gold-Standard-Label")


# ============================================================
# CELL 3: Embeddings extrahieren (mit MobileCLIP-S1 Image Encoder)
# ============================================================
def safe_open(path):
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError) as e:
        print(f"  ⚠ Skipped ({os.path.basename(path)}): {e}")
        return None

def embed(path):
    img = safe_open(path)
    if img is None:
        return None
    tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = model.encode_image(tensor)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().numpy()[0]

X, y, missing = [], [], []
for fname, label in zip(filenames, labels):
    img_path = IMAGE_DIR / fname
    if not img_path.exists():
        missing.append(fname)
        continue
    emb = embed(img_path)
    if emb is None:
        continue
    X.append(emb)
    y.append(label)

if missing:
    print(f"WARNUNG: {len(missing)} Bilder nicht gefunden, z.B.: {missing[:5]}")

X = np.array(X)
y = np.array(y)
print(f"Embeddings extrahiert für {len(X)} Bilder")

np.save("embeddings_298_mobileclip.npy", X)
np.save("labels_298_gold.npy", y)


# ============================================================
# CELL 4: Linear Probe trainieren + 5-fold Cross-Validation
# ============================================================
clf = LogisticRegression(max_iter=1000, C=0.5, class_weight="balanced")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = cross_val_score(clf, X, y, cv=skf, scoring="roc_auc")
print(f"AUC (5-fold CV): {auc_scores.mean():.3f} +/- {auc_scores.std():.3f}")

y_pred = cross_val_predict(clf, X, y, cv=skf)
print(classification_report(y, y_pred))
print("Confusion Matrix (Reihen=tatsächlich, Spalten=vorhergesagt):")
print(confusion_matrix(y, y_pred, labels=["human", "non_human"]))

# Finales Modell auf allen 298 Bildern trainieren (für Einsatz auf restlichem Datensatz)
clf.fit(X, y)
joblib.dump(clf, "linear_probe_mobileclip_human_nonhuman.joblib")
print("Modell gespeichert: linear_probe_mobileclip_human_nonhuman.joblib")

C:\Users\Lukas\miniconda3\envs\kilimanjaro\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Lukas\miniconda3\envs\kilimanjaro\lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Gerät: cuda
Modell geladen: MobileCLIP-S1
Geladen: 298 Bilder mit Gold-Standard-Label
Embeddings extrahiert für 298 Bilder
AUC (5-fold CV): 0.981 +/- 0.014
              precision    recall  f1-score   support

       human       0.82      0.98      0.89        92
   non_human       0.99      0.90      0.94       206

    accuracy                           0.93       298
   macro avg       0.90      0.94      0.92       298
weighted avg       0.94      0.93      0.93       298

Confusion Matrix (Reihen=tatsächlich, Spalten=vorhergesagt):
[[ 90   2]
 [ 20 186]]
Modell gespeichert: linear_probe_mobileclip_human_nonhuman.joblib
